# ⚖️ Lekcja 6: Walidacja krzyżowa `StratifiedKFold` i Wielki Turniej 4 Klasyfikatorów

Mamy już komplet **9 dopracowanych cech** (`Pclass`, `Sex_Num`, `Age`, `Fare`, `Has_Cabin`, `FamilySize`, `IsAlone`, `Title_Num`, `Embarked_Num`). Czas wybrać nasz najlepszy model!

Zanim jednak staniemy na linii startowej, poznajmy dwie nowości charakterystyczne dla **klasyfikacji**:

---

### 1. 🍰 Co oznacza słowo *Stratified* w `StratifiedKFold(n_splits=5)`? (Równe porcje rodzynek w każdym kawałku ciasta!)

W kursie *House Prices* używaliśmy zwykłego `KFold(n_splits=5)`, który po prostu kroił tabelę na 5 równych kawałków.
W klasyfikacji (gdzie na całym statku przeżyło dokładnie **`38.4%`** osób, a zginęło **`61.6%`**) zwykłe losowe krojenie mogłoby sprawić, że do Kawałka nr 1 wpadłoby przypadkiem `50%` ocalałych, a do Kawałka nr 2 tylko `25%` ocalałych!

Dlatego w klasyfikacji zawsze używamy **`StratifiedKFold`** (Walidacji Warstwowej / Stratyfikowanej):
* Wyobraź sobie ciasto z rodzynkami, gdzie rodzynki stanowią `38.4%` ciasta.
* **`StratifiedKFold` kroi naszych 891 pasażerów na 5 części tak, aby w KAŻDYM z 5 kawałków odsetek ocalałych wynosił dokładnie `38.4%`!** Dzięki temu wszystkie 5 egzaminów ma identyczny poziom trudności.

---

### 2. 🧠 Nowy zawodnik w turnieju: Czym jest Regresja Logistyczna (`LogisticRegression`)?

W naszym turnieju obok trzech modeli drzewiastych (**Drzewa Decyzyjnego**, **Lasu Losowego** i **Gradient Boostingu**) wystawimy czwartego, słynnego zawodnika: **Regresję Logistyczną (`LogisticRegression`)**.

> 🤔 **„Zaraz, zaraz! Dlaczego ten model ma w nazwie słowo *Regresja*, skoro używamy go do *Klasyfikacji* (`0` lub `1`)?!”**

To jedno z najbardziej mylących nazwisk w historii statystyki! Zobacz, jak działa ten model pod maską:
1. **Krok 1 (Równanie liniowe — jak w regresji):** Model mnoży każdą cechę pasażera przez odpowiednią wagę (np. dodaje punkty za bycie kobietą i 1. klasę, odejmuje punkty za starszy wiek).
2. **Krok 2 (Funkcja Logistyczna / Sigmoida — kształt litery S):** Następnie przepuszcza ten wynik przez specjalny wzór matematyczny (tzw. *funkcję logistyczną*), który ściska dowolną liczbę do przedziału **od `0.0` (`0%`) do `1.0` (`100%`)** — czyli zamienia ją na **prawdopodobieństwo przeżycia**!
3. **Krok 3 (Decyzja `0` lub `1`):** Jeśli wyliczone prawdopodobieństwo przeżycia wynosi **co najmniej `50%` (`>= 0.50`)**, model mówi **`1` (Przeżył)**; jeśli mniej niż `50%` — mówi **`0` (Zginął)**!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 1. Wczytujemy dane z folderu ../data/ i wykonujemy wszystkie kroki przygotowania cech z Lekcji 3 i 5:
train = pd.read_csv('../data/train.csv')

# A) Wielkość rodziny i samotność
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
train['Has_Cabin'] = train['Cabin'].notnull().astype(int)

# B) Wyciągnięcie tytułu z nazwiska i inteligentne uzupełnienie wieku po tytule
rzadkie_tytuly = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
train['Title'] = train['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
train['Title'] = train['Title'].replace(rzadkie_tytuly, 'Inne')

mediany_wieku = train.groupby('Title')['Age'].median()
train['Age'] = train['Age'].fillna(train['Title'].map(mediany_wieku))
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

# C) Kodowanie słów na liczby
train['Sex_Num'] = train['Sex'].map({'male': 0, 'female': 1})
train['Title_Num'] = train['Title'].map({'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4, 'Inne': 5})
train['Embarked_Num'] = train['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

cechy = ['Pclass', 'Sex_Num', 'Age', 'Fare', 'Has_Cabin', 'FamilySize', 'IsAlone', 'Title_Num', 'Embarked_Num']
X = train[cechy]
y = train['Survived']

# 2. Przygotowujemy sędziego: 5-krotna Walidacja Krzyżowa ze Stratyfikacją (StratifiedKFold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Wystawiamy do turnieju 4 klasyfikatory:
modele = [
    ('1. Drzewo Decyzyjne\n(max_depth=4)', DecisionTreeClassifier(max_depth=4, random_state=42)),
    ('2. Regresja Logistyczna\n(Model liniowy)', LogisticRegression(max_iter=1000, random_state=42)),
    ('3. Las Losowy\n(Random Forest)', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)),
    ('4. Gradient Boosting\n(Sztafeta drzew)', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42))
]

nazwy = []
srednie_acc = []

print('=== 🏆 WYNIKI TURNIEJU KLASYFIKATORÓW (5-FOLD STRATIFIED CV) ===')
for nazwa, model in modele:
    wyniki = cross_val_score(model, X, y, cv=skf, scoring='accuracy') * 100
    nazwy.append(nazwa)
    srednie_acc.append(wyniki.mean())
    czysta = nazwa.replace('\n', ' ')
    print(f'{czysta:42s} -> Średnia dokładność: {wyniki.mean():.2f}%')

# 4. Sprawdzamy też ważność naszych 9 cech w zwycięskim modelu Gradient Boosting
najlepszy_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
najlepszy_model.fit(X, y)
waznosc = pd.Series(najlepszy_model.feature_importances_, index=cechy).sort_values(ascending=True)

# 5. WIZUALIZACJA: Po lewej wyniki 4 modeli, po prawej ranking ważności 9 cech!
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

kolory = ['lightcoral', 'khaki', 'skyblue', 'mediumseagreen']
ax1.bar(nazwy, srednie_acc, color=kolory, edgecolor='black', width=0.55)
ax1.set_title('1. Średnia dokładność (5-Fold Accuracy) — im WIĘCEJ, tym LEPIEJ!')
ax1.set_ylabel('Dokładność — Accuracy (%)')
ax1.set_ylim(75, 87)
for i, v in enumerate(srednie_acc):
    ax1.text(i, v + 0.3, f'{v:.2f}%', ha='center', fontweight='bold', fontsize=11)
ax1.grid(axis='y', linestyle=':', alpha=0.6)

ax2.barh(waznosc.index, waznosc.values, color='steelblue', edgecolor='black')
ax2.set_title('2. Na co najbardziej patrzy nasz model? (Ważność cech)')
ax2.set_xlabel('Udział cechy w decyzjach modelu (feature_importances_)')
ax2.grid(axis='x', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### 🔬 Co mówią nam wyniki turnieju i wykres ważności cech?

1. **Ewolucja dokładności na przestrzeni 4 modeli:**
   * Pojedyncze **Drzewo Decyzyjne (`max_depth=4`)** dzięki naszym nowym cechom z Lekcji 5 podskoczyło z `79.9%` aż do **`81.82%`**!
   * **Regresja Logistyczna (`82.27%`)** pokazuje, że prosty model liniowy oparty na równaniu matematycznym radzi sobie na dobrze przygotowanych cechach lepiej niż pojedyncze drzewo.
   * **Las Losowy (`83.28%`)** — głosowanie 100 drzew daje kolejny procent przewagi.
   * **Gradient Boosting (`84.17%`)** — sztafeta 100 małych drzewek (`max_depth=3`, `learning_rate=0.05`) poprawiających błędy poprzedników znów zgarnia złoty medal, trafnie przewidując los ponad **84 na 100 pasażerów**!
2. **Zwróć uwagę na `scoring='accuracy'` w `cross_val_score`:**
   * Pamiętasz, jak w *House Prices* musieliśmy pisać `'neg_mean_absolute_error'` z minusem na początku, bo błąd dolarowy miał być jak najmniejszy?
   * W klasyfikacji dokładność (`accuracy`) spełnia naturalną zasadę **„im WIĘCEJ procent, tym LEPIEJ”**, dlatego nie ma tu żadnego przedrostka `neg_` ani minusa!
3. **Spójrz na prawy wykres ważności cech:**
   * Na pierwszym miejscu podium stoi nasza nowa cecha **`Title_Num`** (tytuł wyciągnięty z nazwiska, który łączy w sobie płeć oraz odróżnia małych chłopców `Master` od dorosłych mężczyzn `Mr`!), a tuż za nią **`Sex_Num`**, klasa biletu **`Pclass`**, cena biletu **`Fare`**, wielkość rodziny **`FamilySize`** oraz wiek **`Age`**.

In [ ]:
# 🎯 ĆWICZENIE: Zobaczmy, jak model myśli w procentach prawdopodobieństwa (.predict_proba)!
#
# Zwykła metoda `najlepszy_model.predict(X)` zwraca tylko ostateczną decyzję: 0 (Zginął) lub 1 (Przeżył).
# Ale każdy klasyfikator ma też metodę `.predict_proba(X)` (od Probability — prawdopodobieństwo),
# która dla każdego pasażera zwraca DWIE liczby: [Szansa_na_0, Szansa_na_1]!
#
# 1. Wywołaj `szanse = najlepszy_model.predict_proba(X[:5])` dla pierwszych 5 pasażerów.
# 2. Wypisz `np.round(szanse * 100, 1)` za pomocą print() i porównaj z pierwszymi 5 wierszami
#    `train[['Name', 'Pclass', 'Sex', 'Age', 'Survived']].head(5)`!

# Miejsce na Twój kod:


---
### 📌 Podsumowanie lekcji w 3 punktach

1. **Stratyfikacja (`StratifiedKFold`):** W problemach klasyfikacji dzielimy dane na 5 foldów za pomocą `StratifiedKFold`, co gwarantuje, że w każdym z 5 kawałków egzaminacyjnych odsetek ocalałych wynosi dokładnie `38.4%`.
2. **Porównanie 4 rodzin klasyfikatorów:** Na tym samym zestawie 9 cech dokładność rośnie od pojedynczego Drzewa Decyzyjnego (`81.82%`), przez Regresję Logistyczną (`82.27%`) i Las Losowy (`83.28%`), aż po zwycięski **Gradient Boosting (`84.17%`)**.
3. **Prawdopodobieństwo przeżycia (`.predict_proba()`):** Pod maską klasyfikatory nie zgadują w ciemno `0` lub `1`, lecz wyliczają ciągłe prawdopodobieństwo przeżycia (od `0%` do `100%`) i dopiero po przekroczeniu progu `50%` przyznają klasę `1`.